[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/advanced/15_Pipeline_Composition.ipynb)

# Pipeline Composition

## Overview

A pipeline built with `PipelineBuilder.build()` is a reusable unit. `PipelineComposer` takes pipelines you already have and produces a new one, so a stage shared by several workflows is written once and composed in wherever it belongs.

This notebook covers all three composition operations, how step names are kept apart, and how a composed pipeline records where it came from.

## What you'll learn

- `chain()` — run pipelines one after another
- `merge()` — run pipelines as parallel branches, with an optional join step
- `nest()` — splice a sub-pipeline before, after, or in place of a single step
- `PipelineBuilder.include()` — compose while you are still building
- How namespacing keeps colliding step names apart, and how composition lineage survives serialization

## Prerequisites

Python 3.8+ and `semantica`. Every handler in this notebook is a plain Python function, so nothing here needs an LLM API key or an external database.

*Level: Advanced*


In [ ]:
!pip install -qU semantica

In [ ]:
from semantica.pipeline import (
    ExecutionEngine,
    PipelineBuilder,
    PipelineComposer,
    PipelineSerializer,
    PipelineStep,
)
from semantica.utils.exceptions import ValidationError

composer = PipelineComposer()
engine = ExecutionEngine()

## Step 1: Two pipelines worth reusing

Our example turns short filings into relationship triples. We split it into two pipelines that a real project would own separately: one that loads and cleans documents, and one that extracts entities and relations.

Handlers receive the previous step's output as their first argument and return the next step's input.


In [ ]:
DOCUMENTS = [
    {"id": "doc-1", "text": "  Acme Corp acquired Initech in 2021.  "},
    {"id": "doc-2", "text": "Initech partnered with Globex in 2019."},
    {"id": "doc-3", "text": "Globex   acquired Umbrella in 2023."},
]

COMPANIES = ["Acme Corp", "Initech", "Globex", "Umbrella"]
PREDICATES = ["acquired", "partnered with"]


def load_documents(data, **config):
    """Entry step: ignores upstream data and emits the raw corpus."""
    return [dict(doc) for doc in DOCUMENTS]


def clean_documents(docs, **config):
    """Collapse whitespace so the matcher sees predictable text."""
    return [{**doc, "text": " ".join(doc["text"].split())} for doc in docs]


def extract_entities(docs, **config):
    for doc in docs:
        doc["entities"] = [c for c in COMPANIES if c in doc["text"]]
    return docs


def extract_relations(docs, **config):
    triples = []
    for doc in docs:
        for predicate in PREDICATES:
            if predicate not in doc["text"]:
                continue
            head, _, tail = doc["text"].partition(predicate)
            subject = next((c for c in COMPANIES if c in head), None)
            obj = next((c for c in COMPANIES if c in tail), None)
            if subject and obj:
                triples.append(
                    {
                        "subject": subject,
                        "predicate": predicate,
                        "object": obj,
                        "source": doc["id"],
                    }
                )
    return triples


def build_ingest_pipeline(name="ingest"):
    builder = PipelineBuilder()
    builder.add_step("load", "file_ingest", handler=load_documents)
    builder.add_step("clean", "text_normalize", handler=clean_documents)
    builder.connect_steps("load", "clean")
    return builder.build(name)


def build_extract_pipeline(name="extract"):
    builder = PipelineBuilder()
    builder.add_step("entities", "ner_extract", handler=extract_entities)
    builder.add_step("relations", "relation_extract", handler=extract_relations)
    builder.connect_steps("entities", "relations")
    return builder.build(name)


ingest_pipeline = build_ingest_pipeline()
extract_pipeline = build_extract_pipeline()

print("ingest :", [s.name for s in ingest_pipeline.steps])
print("extract:", [s.name for s in extract_pipeline.steps])

## Step 2: `chain()` — run them end to end

`chain()` makes every entry step of each pipeline wait for every terminal step of the one before it, so `extract` starts only once `ingest` has fully finished.


In [ ]:
end_to_end = composer.chain(ingest_pipeline, extract_pipeline, name="filings_to_triples")

for step in end_to_end.steps:
    print(f"{step.name:<20} depends on {step.dependencies}")

result = engine.execute_pipeline(end_to_end)

print("\nsuccess:", result.success)
for triple in result.output:
    print(f"  {triple['subject']} --{triple['predicate']}--> {triple['object']}")

### The source pipelines are untouched

Composition copies every step, so the pipelines you composed are still exactly as you built them — same names, same dependencies, and their run status is unchanged. You can compose the same pipeline into as many workflows as you like.


In [ ]:
print("ingest steps :", [s.name for s in ingest_pipeline.steps])
print("ingest deps  :", [s.dependencies for s in ingest_pipeline.steps])
print("ingest status:", [s.status.value for s in ingest_pipeline.steps])

## Step 3: Step names are namespaced

Two pipelines will often both have a step called `parse` or `load`. Composition prefixes each source pipeline's step names with the pipeline's own name and rewrites the dependencies to match, so a collision cannot happen by accident.

Pass `namespace=False` to keep the original names — composition then raises `ValidationError` if two names collide.


In [ ]:
# Two pipelines that both call their first step "load"
other_ingest = build_ingest_pipeline("newswire")

# Default: each pipeline's steps are prefixed with its own name
namespaced = composer.chain(ingest_pipeline, other_ingest)
print("namespaced :", [s.name for s in namespaced.steps])

# namespace=False keeps the original names and refuses to merge the collision
try:
    composer.chain(ingest_pipeline, other_ingest, namespace=False)
except ValidationError as exc:
    print("\nrejected   :", exc)

# Explicit prefixes, one per pipeline (None means "no prefix for this one")
explicit = composer.chain(ingest_pipeline, other_ingest, namespace=["raw", "wire"])
print("\nexplicit   :", [s.name for s in explicit.steps])

# Composing a pipeline with itself works: the second copy gets a _2 suffix
twice = composer.chain(ingest_pipeline, ingest_pipeline)
print("self       :", [s.name for s in twice.steps])

## Step 4: `merge()` — parallel branches with a join

`merge()` places pipelines side by side without adding dependencies between them, so they stay independent branches of one graph and `ParallelismManager` is free to run them concurrently. A `join` step waits for every branch.

One thing to know about the engine: `ExecutionEngine` threads a *single* value along the topological order, so a join step does not automatically receive both branches' outputs as its input. When branches need to hand results to a join, have each branch write to a shared sink — that is what `COLLECTED` does below.


In [ ]:
COLLECTED = {}


def make_branch_handler(source_name, documents):
    """Terminal handler for a branch: extract triples and record them."""

    def handler(data, **config):
        docs = clean_documents([dict(doc) for doc in documents])
        triples = extract_relations(docs)
        COLLECTED[source_name] = triples
        return triples

    return handler


def merge_triples(data, **config):
    """Join step: fold every branch's triples into one deduplicated list."""
    seen, merged = set(), []
    for source_name in sorted(COLLECTED):
        for triple in COLLECTED[source_name]:
            key = (triple["subject"], triple["predicate"], triple["object"])
            if key not in seen:
                seen.add(key)
                merged.append(triple)
    return merged


def build_source_pipeline(name, documents):
    builder = PipelineBuilder()
    builder.add_step("fetch", "http_ingest", handler=lambda data, **c: documents)
    builder.add_step(
        "extract", "relation_extract", handler=make_branch_handler(name, documents)
    )
    builder.connect_steps("fetch", "extract")
    return builder.build(name)


filings = build_source_pipeline("filings", DOCUMENTS[:2])
newswire = build_source_pipeline("newswire", DOCUMENTS[1:])

merged = composer.merge(
    filings,
    newswire,
    name="multi_source",
    join=PipelineStep(name="merge_triples", step_type="kg_merge", handler=merge_triples),
)

for step in merged.steps:
    print(f"{step.name:<20} depends on {step.dependencies}")

COLLECTED.clear()
result = ExecutionEngine(max_workers=2).execute_pipeline(merged)

print("\nbranches collected:", {k: len(v) for k, v in COLLECTED.items()})
print("merged triples    :", len(result.output))
for triple in result.output:
    print(f"  {triple['subject']} --{triple['predicate']}--> {triple['object']}")

The two `fetch` steps have no dependencies, so they are both entry points; the join step depends on each branch's terminal step. The join step is never namespaced — it belongs to the composition, not to any single source.

## Step 5: `nest()` — splice a sub-pipeline into one position

`nest()` inserts a pipeline at a single step of another one. Only the child is namespaced: the parent keeps its own step names, so you can nest the same sub-pipeline at several positions under different prefixes.

To see the rewiring clearly, we use a parent that fans out and back in.


In [ ]:
def passthrough(label):
    def handler(data, **config):
        return data

    return handler


builder = PipelineBuilder()
for step_name in ("load", "left", "right", "store"):
    builder.add_step(step_name, "stage", handler=passthrough(step_name))
builder.connect_steps("load", "left")
builder.connect_steps("load", "right")
builder.connect_steps("left", "store")
builder.connect_steps("right", "store")
fanout = builder.build("fanout")

validate = build_extract_pipeline("validate")


def show(title, pipeline):
    print(title)
    for step in pipeline.steps:
        print(f"  {step.name:<20} depends on {step.dependencies}")
    print()


show("parent as built:", fanout)

# "after" (default): everything that depended on "load" now waits for the child
show('nest(..., at="load", mode="after"):', composer.nest(fanout, validate, at="load"))

# "before": the child inherits the anchor's dependencies, the anchor waits for it
show(
    'nest(..., at="store", mode="before"):',
    composer.nest(fanout, validate, at="store", mode="before"),
)

# "replace": the anchor is removed and both sides are rewired around the child
show(
    'nest(..., at="left", mode="replace"):',
    composer.nest(fanout, validate, at="left", mode="replace"),
)

Because only the child is namespaced, the same sub-pipeline can be nested more than once under different prefixes.


In [ ]:
once = composer.nest(fanout, validate, at="left", namespace="check_left")
twice = composer.nest(once, validate, at="right", namespace="check_right")

print([s.name for s in twice.steps])

## Step 6: `include()` — compose while you build

`PipelineBuilder.include()` is the DSL-side equivalent: it drops a built pipeline into a builder you are still assembling. It returns the builder, so it chains with `connect_steps()` and `set_parallelism()`.

Included steps carry the namespace prefix, so wire downstream steps to the prefixed name.


In [ ]:
def summarize(triples, **config):
    return {
        "triple_count": len(triples),
        "subjects": sorted({t["subject"] for t in triples}),
    }


builder = PipelineBuilder()
builder.include(ingest_pipeline)
builder.include(extract_pipeline, after="ingest.clean")
builder.add_step("summary", "report", handler=summarize)
builder.connect_steps("extract.relations", "summary")

assembled = builder.build("filings_report")

for step in assembled.steps:
    print(f"{step.name:<20} depends on {step.dependencies}")

print("\nreport:", ExecutionEngine().execute_pipeline(assembled).output)

## Step 7: Lineage and serialization

Every composed pipeline records where it came from in `metadata["composition"]`. The record is plain data, so it survives a `PipelineSerializer` round trip along with the step graph.

Handler functions are *not* serializable, so re-attach them on the restored pipeline before executing — the dependency graph, however, comes back intact.


In [ ]:
import json

print(json.dumps(end_to_end.metadata["composition"], indent=2))

serializer = PipelineSerializer()
restored = serializer.deserialize_pipeline(serializer.serialize_pipeline(end_to_end))

print("\nrestored dependencies:")
for step in restored.steps:
    print(f"  {step.name:<20} depends on {step.dependencies}")

# Re-attach handlers by step type, then run the restored pipeline
HANDLERS = {
    "file_ingest": load_documents,
    "text_normalize": clean_documents,
    "ner_extract": extract_entities,
    "relation_extract": extract_relations,
}
for step in restored.steps:
    step.handler = HANDLERS[step.step_type]

print("\nrestored run:", ExecutionEngine().execute_pipeline(restored).output)

## What composition checks for you

Every composition validates the result with `PipelineValidator` before returning it, so these fail loudly instead of producing a pipeline that misbehaves at run time:

| Problem | Result |
| --- | --- |
| Two steps end up with the same name | `ValidationError` naming the duplicates |
| A dependency points at a step that isn't there | `ValidationError` from validation |
| A source pipeline's steps form a cycle | `ValidationError` — no entry or terminal step |
| A source pipeline has no steps | `ValidationError` |
| `nest()` given an `at` step the parent doesn't have | `ValidationError` |

Pass `validate=False` when you are deliberately assembling a pipeline in stages and know it is still incomplete.

## Summary

- `chain()`, `merge()`, and `nest()` build a new pipeline out of pipelines you already have
- Source pipelines are copied, never mutated, so one sub-pipeline can serve many workflows
- Step names are namespaced by their source pipeline and dependencies are rewritten to match
- `PipelineBuilder.include()` brings the same capability into the DSL
- Composition lineage lands in `metadata["composition"]` and survives serialization

## Next steps

- [Pipeline Builder guide](https://github.com/semantica-agi/semantica/blob/main/docs/guides/pipeline.md) — retry policies, parallelism, delta mode, and progress tracking
- [Multi-Source Data Integration](https://github.com/semantica-agi/semantica/blob/main/cookbook/advanced/06_Multi_Source_Data_Integration.ipynb) — merging the graphs that these branches produce
- [Reasoning and Inference](https://github.com/semantica-agi/semantica/blob/main/cookbook/advanced/08_Reasoning_and_Inference.ipynb) — inferring new facts from the triples above
